In [6]:
from qdrant_client.models import PointStruct

In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

client = QdrantClient(host="localhost", port=6333)

# Creating collections to insert points into
client.recreate_collection(
    collection_name="table",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

/var/folders/9b/87_d8hyd7mq1rhhh_fbqcgj80000gn/T/ipykernel_23646/3187946990.py:4: UserWarning: Qdrant client version 1.17.0 is incompatible with server version 1.15.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(host="localhost", port=6333)
/var/folders/9b/87_d8hyd7mq1rhhh_fbqcgj80000gn/T/ipykernel_23646/3187946990.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [3]:
client.recreate_collection(
    collection_name="table-column",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

/var/folders/9b/87_d8hyd7mq1rhhh_fbqcgj80000gn/T/ipykernel_23646/2551528071.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [ ]:
import json

# Load the Schema which has context
with open('context.json', 'r') as f:
    data = json.load(f)

In [ ]:
from sentence_transformers import SentenceTransformer
# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/ayushmayekar/miniforge3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Create a qdrant client to interact with
client = QdrantClient(host="localhost", port=6333)

/var/folders/9b/87_d8hyd7mq1rhhh_fbqcgj80000gn/T/ipykernel_23646/109858654.py:1: UserWarning: Qdrant client version 1.17.0 is incompatible with server version 1.15.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(host="localhost", port=6333)


In [15]:
points = []
points_col = []

col_id = 0  # separate counter for columns

for idx, (key, value) in enumerate(data.items()):
    # ---- TABLE LEVEL ----
    table_text = f"""
    Table: {key}.
    Description: {value['table_description']}.
    """

    table_vector = model.encode(table_text).tolist()

    points.append(
        PointStruct(
            id=idx,
            vector=table_vector,
            payload={
                "table": key,
                "description": value["table_description"],
                "primary_keys": value["primary_keys"],
                "foreign_keys": value["foreign_keys"]
            }
        )
    )

    # ---- COLUMN LEVEL ----
    for col in value["columns"]:
        col_text = f"""
        Column: {col['column_name']}.
        Table: {key}.
        Description: {col['description']}.
        Data type: {col['data_type']}.
        """

        col_vector = model.encode(col_text).tolist()

        points_col.append(
            PointStruct(
                id=col_id,
                vector=col_vector,
                payload={
                    "table": key,
                    "column": col["column_name"],
                    "datatype": col["data_type"],
                    "indexed": col["is_indexed"],
                    "description": col["description"]
                }
            )
        )

        col_id += 1

client.upsert(collection_name="table", points=points)
client.upsert(collection_name="table-column", points = points_col)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [17]:
from qdrant_client.models import PayloadSchemaType

In [18]:
# creating an index for the field table in collection of columns to apply filters faster
client.create_payload_index(
    collection_name="table-column",
    field_name="table",
    field_schema=PayloadSchemaType.KEYWORD
)


UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)